# Chapter 12 — Advanced Visualization
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jzazueta/ds-business-economics/blob/main/notebooks/ch12_visualization.ipynb)

*Data Science for Business and Economics: A Python Approach* — Dr. Jorge Zazueta Gutiérrez

This notebook builds every figure in Chapter 12 from real INEGI data: the DENUE business directory
(May 2026 release), the 2020 population census, the 2017 geostatistical framework, and the monthly
retail survey (EMEC). The static figures are drawn at the book's print size and saved to `figures/`.
The plotly figures run live in the notebook — hover, zoom, press the buttons — and can also be
saved as standalone HTML files to share.

The data comes from a snapshot stored in the book repository (`data/` and `geo/`). The first code
cell after the setup finds those files locally or downloads them from the repository, so the notebook
runs in Colab with nothing checked out. `scripts/build_ch12_snapshot.py` rebuilds that snapshot
from the original sources. One cell calls the live DENUE API; it needs a free INEGI token and falls
back to the snapshot when no token is set.

In [ ]:
book_metadata = {
    "title": "Data Science for Business and Economics: A Python Approach",
    "chapter": 12,
    "chapter_title": "Advanced Visualization",
    "author": "Jorge Zazueta Gutierrez",
    "protagonist": "Andrés Villaseñor",
    "part": 2,
    "part_title": "Methods and Models",
    "requires": ["pandas", "numpy", "matplotlib", "seaborn", "geopandas", "scipy",
                 "plotly==5.24.1", "kaleido==0.2.1", "colorspacious", "requests"],
    "notebook_version": "0.1",
}
book_metadata

## Setup

In [ ]:
# Pinned versions: plotly 5 exports static images with the kaleido build that ships its own browser.
# (plotly 6+ needs a separately installed Chrome for static export.)
%pip install -q plotly==5.24.1 kaleido==0.2.1 geopandas colorspacious

In [ ]:
from pathlib import Path
import requests

# Use the repository's data/ and geo/ folders when they are present (a local clone);
# otherwise download each file from the book repository on GitHub.
REPO_RAW = "https://raw.githubusercontent.com/jzazueta/ds-business-economics/main"
FILES = {
    "data": ["minisupers.csv", "oxxo_mun_harmonized.csv", "oxxo_state.csv", "mun_pop_2020.csv",
             "crosswalk_new_municipalities.csv", "emec_retail_monthly.csv",
             "macro_food_away_from_home_spending.csv", "macro_food_away_from_home_cpi.csv"],
    "geo": ["mx_states_2017.geojson", "mx_municipalities_2017.geojson"],
}
REPO = Path("..") if Path("../data").exists() else Path(".")
for folder, names in FILES.items():
    (REPO / folder).mkdir(exist_ok=True)
    for name in names:
        path = REPO / folder / name
        if not path.exists():
            response = requests.get(f"{REPO_RAW}/{folder}/{name}", timeout=120)
            response.raise_for_status()
            path.write_bytes(response.content)
            print(f"downloaded {folder}/{name}")
DATA, GEO, FIGURES = REPO / "data", REPO / "geo", REPO / "figures"
FIGURES.mkdir(exist_ok=True)

In [ ]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Patch, Rectangle
import seaborn as sns

# Print-size defaults: figures are drawn 6 inches wide (the book's text width), so text is set
# in points that stay readable on the page instead of being shrunk later by LaTeX.
sns.set_theme(style="white")
plt.rcParams.update({
    "font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7.5,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 300, "savefig.bbox": "tight",
})
WIDTH = 6.0   # inches

# Book palette: steel blue and gray for single-focus charts; Okabe-Ito for categories;
# a steel-blue ramp for the four ordered regions (north to south).
STEEL, GRAY, LIGHT = "#4682B4", "#8C8C8C", "#D9D9D9"
CHAIN = {"OXXO": "#D55E00", "3B": "#0072B2", "7-Eleven": "#009E73", "Neto": "#E69F00",
         "Circle K": "#CC79A7", "BA Express": "#56B4E9", "Other": LIGHT}
REGIONS = ["Norte", "Centro Norte", "Centro", "Sur"]
REGION_COLOR = {"Norte": "#132B45", "Centro Norte": "#3B6E9F", "Centro": "#7FA8D0", "Sur": "#B7CFE6"}

# Short state names for labels
SHORT = {"Coahuila de Zaragoza": "Coahuila", "Veracruz de Ignacio de la Llave": "Veracruz",
         "Michoacán de Ocampo": "Michoacán", "México": "Estado de México"}
short = lambda s: SHORT.get(s, s)

def save(fig, name):
    # Save a matplotlib figure to figures/ at print resolution
    fig.savefig(FIGURES / f"{name}.png")

## 12.1 The Question and the First Map

In [ ]:
import os

def get_denue_token():
    # The DENUE token: environment variable first, then Colab's Secrets panel.
    # Register for a free token at https://www.inegi.org.mx/servicios/api_denue.html
    token = os.environ.get("DENUE_TOKEN", "")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("DENUE_TOKEN") or ""
        except Exception:
            pass
    return token

In [ ]:
import requests

DENUE_API = ("https://www.inegi.org.mx/app/api/denue/v1"
             "/consulta")

def denue_search(term, state_code, token,
                 first=1, last=1000):
    # BuscarEntidad: establishments whose name matches
    # `term` in one state, records first..last
    url = (f"{DENUE_API}/BuscarEntidad/{term}/{state_code}"
           f"/{first}/{last}/{token}")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    try:
        return pd.DataFrame(response.json())
    except ValueError:   # not JSON: INEGI's error text
        raise RuntimeError(response.text.strip())

# get_denue_token() reads the token from an environment
# variable or Colab's Secrets panel; 24 is San Luis Potosi
token = get_denue_token()
if token:
    live = denue_search("oxxo", "24", token)
    print(f"{len(live)} records for 'oxxo' in state 24")
else:
    print("No DENUE token set; using the snapshot instead.")

In [ ]:
# The snapshot: every minisupermarket in DENUE, with its chain
minisupers = pd.read_csv(DATA / "minisupers.csv",
                         dtype={"cve_ent": str, "key": str})
print(f"{len(minisupers):,} minisupermarkets")
print(minisupers["chain"].value_counts())

In [ ]:
# OXXO stores and 2020 population, by state
oxxo_state = (pd.read_csv(DATA / "oxxo_state.csv")
              .astype({"stores": int}))
# Stores per 10,000 residents
oxxo_state["per10k"] = (oxxo_state["stores"]
                        / oxxo_state["pop"] * 1e4)
stores = oxxo_state["stores"].sum()
people = oxxo_state["pop"].sum()
national = stores / people * 1e4
print(f"{stores:,} OXXO stores, {people:,} residents")
print(f"National rate: {national:.2f} per 10,000 residents")

In [ ]:
# Notebook only: state codes and short names, used by the figure cells
state_names = (pd.read_csv(DATA / "mun_pop_2020.csv", dtype=str)[["ENTIDAD", "NOM_ENT"]]
               .drop_duplicates().rename(columns={"ENTIDAD": "cve_ent"}))
oxxo_state = oxxo_state.merge(state_names, on="NOM_ENT")
oxxo_state["name"] = oxxo_state["NOM_ENT"].map(short)
NATIONAL = national

In [ ]:
# State boundaries, joined to the OXXO table by state name;
# EPSG 6372 is INEGI's map projection, in meters
states = gpd.read_file(GEO / "mx_states_2017.geojson")
state_map = (states.merge(oxxo_state, on="NOM_ENT")
             .to_crs(6372))

fig, axes = plt.subplots(1, 2, figsize=(6, 2.6))
panels = [(axes[0], "stores", "OXXO stores"),
          (axes[1], "per10k",
           "OXXO stores per 10,000 residents")]
for ax, column, label in panels:
    # cividis: dark blue for low values, yellow for high
    state_map.plot(column=column, cmap="cividis", ax=ax,
                   edgecolor="white", linewidth=0.3,
                   legend=True,
                   legend_kwds={"orientation": "horizontal",
                                "label": label})
    ax.set_axis_off()

In [ ]:
# State map: raw store counts (a) against stores per 10,000 residents (b)
state_map = states.merge(oxxo_state, left_on="CVE_ENT", right_on="cve_ent")
state_map["geometry"] = state_map.to_crs(6372).geometry.simplify(2000)
state_map = state_map.set_geometry("geometry").set_crs(6372, allow_override=True)

fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.6))
for ax, column, label in [(axes[0], "stores", "OXXO stores"),
                          (axes[1], "per10k", "OXXO stores per 10,000 residents")]:
    state_map.plot(column=column, cmap="cividis", ax=ax, edgecolor="white", linewidth=0.3, legend=True,
                   legend_kwds={"orientation": "horizontal", "shrink": 0.7, "pad": 0.02, "label": label})
    ax.set_axis_off()
for ax, tag in zip(axes, ["(a)", "(b)"]):
    ax.text(0.02, 0.08, tag, transform=ax.transAxes, fontsize=7.5)
save(fig, "fig12_state_count_vs_percapita")

In [ ]:
# Rank the states twice: by store count and by rate
def rank_of(column):
    return (oxxo_state[column]
            .rank(ascending=False, method="min").astype(int))

oxxo_state["rank_count"] = rank_of("stores")
oxxo_state["rank_rate"] = rank_of("per10k")
columns = ["NOM_ENT", "stores", "rank_count", "per10k",
           "rank_rate"]
for order, n in [("rank_count", 6), ("rank_rate", 3)]:
    table = oxxo_state.sort_values(order)[columns].head(n)
    print(table.round(2).to_string(index=False), end="\n\n")

## 12.2 Rates and their Denominators

In [ ]:
# One row per municipality: OXXO stores and 2020 population
oxxo_mun = (pd.read_csv(DATA / "oxxo_mun_harmonized.csv",
                        dtype={"key": str})
            .astype({"stores": int}))
oxxo_mun["per10k"] = (oxxo_mun["stores"]
                      / oxxo_mun["pop"] * 1e4)

top = oxxo_mun.nlargest(5, "per10k")
print(top[["NOM_MUN", "NOM_ENT", "pop", "stores", "per10k"]]
      .round(2).to_string(index=False))

In [ ]:
from scipy.stats import poisson

# If every municipality had the national rate, how many
# stores would each one have? The count would vary by chance
# around this expected number, following the Poisson
# distribution.
expected = national / 1e4 * oxxo_mun["pop"]
stores = oxxo_mun["stores"]

# Two-sided test at 5%: is the observed count in either tail?
too_few = poisson.cdf(stores, expected) < 0.025
too_many = poisson.sf(stores - 1, expected) < 0.025
oxxo_mun["differs"] = too_few | too_many

bands = pd.cut(oxxo_mun["pop"],
               [0, 10_000, 20_000, np.inf], right=False,
               labels=["under 10k", "10k-20k", "20k+"])
summary = oxxo_mun.groupby(bands, observed=True)["differs"]
print(summary.agg(["size", "sum", "mean"]).round(3))

In [ ]:
from scipy.stats import chi2

FLOOR = 20_000
rate_per_resident = NATIONAL / 1e4
above = oxxo_mun["pop"] >= FLOOR

def chance_band(population, level):
    # Exact Poisson limits for the rate if every municipality had the national rate
    expected = rate_per_resident * population
    alpha = 1 - level
    low = np.where(expected > 0, chi2.ppf(alpha / 2, 2 * expected) / 2, 0)
    high = chi2.ppf(1 - alpha / 2, 2 * expected + 2) / 2
    return low / population * 1e4, high / population * 1e4

fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.5), gridspec_kw={"width_ratios": [1.35, 1]})
ax = axes[0]
x = np.logspace(np.log10(700), np.log10(2.2e6), 400)
for level, alpha in [(0.998, 0.12), (0.95, 0.25)]:             # light and dark chance bands
    low, high = chance_band(x, level)
    ax.fill_between(x, low, high, color=STEEL, alpha=alpha, lw=0)
for k, y_label in [(1, 9.0), (2, 6.2), (3, 4.6)]:             # municipalities with exactly k stores
    ax.plot(x, k * 1e4 / x, ls=(0, (2, 2)), color="#555555", lw=0.6)
    ax.text(k * 1e4 / y_label * 1.12, y_label, f"{k} store{'s' * (k > 1)}", fontsize=5.5, color="#444444",
            bbox=dict(fc="white", ec="none", pad=0.4, alpha=0.85))
ax.scatter(oxxo_mun.loc[~above, "pop"], oxxo_mun.loc[~above, "per10k"], s=3, color=GRAY, alpha=0.45, lw=0)
ax.scatter(oxxo_mun.loc[above, "pop"], oxxo_mun.loc[above, "per10k"], s=4, color=STEEL, alpha=0.8, lw=0)
ax.axhline(NATIONAL, color="black", lw=0.6)
ax.text(1.02, (NATIONAL + 0.3) / 13.8, f"national\n{NATIONAL:.2f}", transform=ax.transAxes, fontsize=5.5, va="center")
ax.axvline(FLOOR, color="black", lw=0.8, ls="--")
ax.text(FLOOR * 1.1, 12.9, "20,000 floor", fontsize=6)
ax.set(xscale="log", xlim=(700, 2.2e6), ylim=(-0.3, 13.5),
       xlabel="Municipal population, 2020 (log scale)", ylabel="OXXO stores per 10,000 residents")
ax.set_xticks([1e3, 1e4, 1e5, 1e6], ["1,000", "10,000", "100,000", "1,000,000"])

ax = axes[1]                                                    # what each floor keeps
floors = np.unique(np.r_[np.logspace(3, 5, 300), FLOOR])
kept_pop = [100 * oxxo_mun.loc[oxxo_mun["pop"] >= f, "pop"].sum() / oxxo_mun["pop"].sum() for f in floors]
kept_units = [100 * (oxxo_mun["pop"] >= f).mean() for f in floors]
ax.plot(floors, kept_pop, color=STEEL, lw=1.5)
ax.plot(floors, kept_units, color=GRAY, lw=1.5)
ax.text(1.1e3, 103, "population covered", color=STEEL, fontsize=6, va="bottom")
ax.text(1.1e3, 84, "municipalities shown", color=GRAY, fontsize=6, va="top")
i = np.where(floors == FLOOR)[0][0]
for value, color in [(kept_pop[i], STEEL), (kept_units[i], GRAY)]:
    ax.plot(FLOOR, value, "o", color=color, ms=4)
    ax.text(FLOOR * 1.12, value + 2.5, f"{value:.0f}%", color=color, fontsize=6, fontweight="bold")
ax.axvline(FLOOR, color="black", lw=0.8, ls="--")
ax.set(xscale="log", ylim=(0, 110), xlim=(1e3, 1e5), xlabel="Population floor (log scale)", ylabel="Percent retained")
ax.set_xticks([1e3, 1e4, 1e5], ["1,000", "10,000", "100,000"])
for a, tag in zip(axes, ["(a)", "(b)"]):
    a.text(-0.14, 1.03, tag, transform=a.transAxes, fontsize=7.5)
fig.tight_layout()
save(fig, "fig12_floor_criterion")

In [ ]:
# What each population floor keeps, and how far one store
# moves the rate of a municipality right at the floor
rows = []
for floor in [5_000, 10_000, 20_000, 50_000]:
    kept = oxxo_mun["pop"] >= floor
    share = (oxxo_mun.loc[kept, "pop"].sum()
             / oxxo_mun["pop"].sum())
    rows.append({"floor": floor,
                 "municipalities": kept.sum(),
                 "pct_population": 100 * share,
                 "one_store_moves": 1e4 / floor})
print(pd.DataFrame(rows).round(2).to_string(index=False))

In [ ]:
# Municipal maps without a floor (a) and with one (b)
FLOOR = 20_000
munis = gpd.read_file(GEO / "mx_municipalities_2017.geojson")
muni_map = (munis[["key", "geometry"]]
            .merge(oxxo_mun, on="key").to_crs(6372))
scale = dict(cmap="cividis", vmin=0, vmax=6, linewidth=0)

fig, axes = plt.subplots(1, 2, figsize=(6, 2.6))
muni_map.plot(column="per10k", ax=axes[0], **scale)
shown = muni_map["pop"] >= FLOOR
muni_map[shown].plot(column="per10k", ax=axes[1], **scale)
muni_map[~shown].plot(color="#E6E6E6", ax=axes[1])  # gray
for ax in axes:
    ax.set_axis_off()

In [ ]:
munis_m, states_m = munis.to_crs(6372), states.to_crs(6372)

# Municipal map with no floor (a) and with the 20,000 floor (b), on one color scale
muni_map = munis_m.merge(oxxo_mun[["key", "pop", "stores", "per10k"]], on="key")
state_lines = states_m.boundary.simplify(1500)
norm, cmap, MASK = mpl.colors.Normalize(0, 6), mpl.colormaps["cividis"], "#E6E6E6"

fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.6))
for ax, floor in [(axes[0], 0), (axes[1], FLOOR)]:
    shown = muni_map["pop"] >= floor
    muni_map[shown].plot(column="per10k", cmap=cmap, norm=norm, ax=ax, lw=0)
    if (~shown).any():
        muni_map[~shown].plot(color=MASK, ax=ax, lw=0)
    state_lines.plot(ax=ax, color="white", lw=0.25)
    ax.set_axis_off()
for ax, tag in zip(axes, ["(a)", "(b)"]):
    ax.text(0.02, 0.08, tag, transform=ax.transAxes, fontsize=7.5)
cax = fig.add_axes([0.2, 0.1, 0.36, 0.03])
bar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap), cax=cax, orientation="horizontal", extend="max")
bar.set_label("OXXO stores per 10,000 residents"); bar.outline.set_linewidth(0.3)
fig.legend(handles=[Patch(fc=MASK, ec="#BBBBBB", lw=0.4, label="population under 20,000")],
           loc="center left", bbox_to_anchor=(0.62, 0.115), frameon=False)
fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.2, wspace=0.02)
save(fig, "fig12_municipal_floor")

## 12.3 Graphical Integrity

In [ ]:
# The same rates as a sorted bar chart with value labels.
# Color marks the states whose rank moves 15 places or more
# between the count and rate rankings (Code Example 12.5).
gap = oxxo_state["rank_count"] - oxxo_state["rank_rate"]
ordered = (oxxo_state.assign(moved=gap.abs() >= 15)
           .sort_values("per10k").reset_index(drop=True))
colors = ordered["moved"].map({True: "#4682B4",
                               False: "#C9C9C9"})

fig, ax = plt.subplots(figsize=(4, 4.8))
ax.barh(ordered.index, ordered["per10k"], color=colors)
ax.axvline(national, color="#444444", lw=0.7, ls="--")
for i, value in enumerate(ordered["per10k"]):
    ax.text(value + 0.05, i, f"{value:.2f}", va="center")
ax.set_yticks(ordered.index, ordered["NOM_ENT"])
ax.set_xticks([])               # labels replace the axis
ax.spines[["bottom", "left"]].set_visible(False)

In [ ]:
# The same variable twice: common defaults (a) and a chart built to be read (b)
# Highlight the states whose rank moves 15 places or more between the two rankings
ranked = oxxo_state.assign(r_count=oxxo_state.stores.rank(ascending=False, method="min"),
                           r_rate=oxxo_state.per10k.rank(ascending=False, method="min"))
highlight = set(ranked.loc[(ranked.r_count - ranked.r_rate).abs() >= 15, "name"])

fig = plt.figure(figsize=(WIDTH, 7.4))
with plt.style.context("ggplot"):                        # the "before": ggplot defaults
    ax = fig.add_axes([0.1, 0.72, 0.88, 0.25])
    alphabetical = oxxo_state.sort_values("name")
    ax.bar(alphabetical.name, alphabetical.per10k, color=plt.get_cmap("tab20")(np.arange(32) % 20),
           edgecolor="black", lw=0.5)
    ax.set_ylabel("OXXO per 10k")
    ax.tick_params(axis="x", rotation=90, labelsize=5.5)
    for spine in ax.spines.values():
        spine.set_visible(True); spine.set_color("black")
    ax.text(0, 1.04, "(a) Before", transform=ax.transAxes, fontsize=7.5)

ax = fig.add_axes([0.24, 0.02, 0.66, 0.55])              # the "after"
ordered = oxxo_state.sort_values("per10k").reset_index(drop=True)
ax.axvline(NATIONAL, color="#444444", lw=0.7, ls=(0, (3, 2)), zorder=0)
ax.text(NATIONAL + 0.04, 31.6, f"national {NATIONAL:.2f}", fontsize=6, va="bottom")
ax.barh(ordered.index, ordered.per10k, height=0.72, zorder=2,
        color=[STEEL if n in highlight else "#C9C9C9" for n in ordered.name])
for i, row in ordered.iterrows():                        # value labels replace the x-axis
    ax.text(row.per10k + 0.05, i, f"{row.per10k:.2f}", va="center", fontsize=5.8, zorder=3,
            color="#333333" if row["name"] in highlight else "#888888",
            path_effects=[pe.withStroke(linewidth=2, foreground="white")])
ax.set_yticks(ordered.index, ordered.name, fontsize=6)
for label, name in zip(ax.get_yticklabels(), ordered.name):
    label.set_color("black" if name in highlight else "#777777")
ax.set(xlim=(0, 4.6), ylim=(-0.7, 32.5), xticks=[])
ax.spines[["bottom", "left"]].set_visible(False); ax.tick_params(axis="y", length=0)
fig.text(0.1, 0.605, "(b) After", fontsize=7.5)
fig.text(0.1, 0.585, "OXXO stores per 10,000 residents, 2026", fontsize=6.5, color="#555555")
save(fig, "fig12_data_ink")

In [ ]:
# Chapter 5's two series: spending and the price index
def read_macro(name):
    path = DATA / f"macro_food_away_from_home_{name}.csv"
    return pd.read_csv(path, parse_dates=["date"])

spending, prices = read_macro("spending"), read_macro("cpi")
macro = spending.merge(prices, on="date")

# Rebase both to 100 in the first month, so they share a scale
for column in ["spending_index", "cpi_food_away"]:
    macro[column + "_100"] = (100 * macro[column]
                              / macro[column].iloc[0])
last = macro.iloc[-1]
s, p = last["spending_index_100"], last["cpi_food_away_100"]
first = macro["date"].iloc[0]
print(f"{first:%b %Y} to {last['date']:%b %Y}")
print(f"Spending {s - 100:+.1f}%, prices {p - 100:+.1f}%")
print(f"Spending in real terms {100 * (s / p - 1):+.1f}%")

In [ ]:
# One axis, one starting point: the gap is now visible
fig, ax = plt.subplots(figsize=(3.2, 2.6))
ax.plot(macro["date"], macro["spending_index_100"],
        color="#4682B4", label="Spending")
ax.plot(macro["date"], macro["cpi_food_away_100"],
        color="#8C8C8C", ls="--", label="Prices")
ax.fill_between(macro["date"], macro["spending_index_100"],
                macro["cpi_food_away_100"], color="#8C8C8C",
                alpha=0.12, lw=0)
ax.axhline(100, color="#BBBBBB", lw=0.6)
ax.set_ylabel("Index, first month = 100")

In [ ]:
macro["spending_100"] = macro["spending_index_100"]
macro["prices_100"] = macro["cpi_food_away_100"]
last = macro.iloc[-1]

import matplotlib.dates as mdates

fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.6), gridspec_kw={"width_ratios": [1, 1.12]})
ax, ax_right = axes[0], axes[0].twinx()                   # (a) the dual-axis chart from Chapter 5
ax.plot(macro.date, macro.spending_index, color=STEEL, lw=1.2)
ax_right.plot(macro.date, macro.cpi_food_away, color=GRAY, lw=1.2, ls="--")
ax.set_ylabel("Spending index", color=STEEL); ax_right.set_ylabel("Price index", color=GRAY)
ax.tick_params(axis="y", colors=STEEL); ax_right.tick_params(axis="y", colors=GRAY)
ax_right.spines["right"].set_visible(True)

ax = axes[1]                                              # (b) both series from the same starting point
ax.plot(macro.date, macro.spending_100, color=STEEL, lw=1.2)
ax.plot(macro.date, macro.prices_100, color=GRAY, lw=1.2, ls="--")
ax.fill_between(macro.date, macro.spending_100, macro.prices_100, color=GRAY, alpha=0.12, lw=0)
ax.axhline(100, color="#BBBBBB", lw=0.6)
x_end = last.date + pd.Timedelta(days=25)
ax.text(x_end, last.prices_100, f"Prices\n+{last.prices_100 - 100:.1f}%", color="#555555", fontsize=6.5, va="center")
ax.text(x_end, last.spending_100, f"Spending\n+{last.spending_100 - 100:.1f}%", color=STEEL, fontsize=6.5, va="center")
ax.set_xlim(macro.date.iloc[0] - pd.Timedelta(days=30), last.date + pd.Timedelta(days=260))
ax.set_ylabel("Index, July 2023 = 100")
for a, tag in zip(axes, ["(a) Two axes", "(b) One axis, common base"]):
    a.set_xticks(pd.to_datetime(["2024-01-01", "2025-01-01", "2026-01-01"]))
    a.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    a.text(0, 1.05, tag, transform=a.transAxes, fontsize=7.5)
fig.tight_layout(w_pad=2.0)
save(fig, "fig12_dual_axis_before_after")

## 12.4 Color

In [ ]:
# Okabe-Ito colors for the chains; light gray for the rest
CHAIN = {"OXXO": "#D55E00", "3B": "#0072B2",
         "7-Eleven": "#009E73", "Neto": "#E69F00",
         "Circle K": "#CC79A7", "BA Express": "#56B4E9",
         "Other": "#D9D9D9"}

# matplotlib's default colors, assigned in the same order
tab10 = plt.get_cmap("tab10").colors
order = (3, 0, 2, 1, 4, 5, 7)     # red for OXXO, and so on
DEFAULT = dict(zip(CHAIN, [tab10[i] for i in order]))

In [ ]:
from itertools import combinations
from colorspacious import cspace_convert

DEUTERANOPIA = {"name": "sRGB1+CVD",
                "cvd_type": "deuteranomaly", "severity": 100}

def seen_by_deuteranope(color):
    # The color a reader with deuteranopia perceives
    rgb = np.array(mpl.colors.to_rgb(color))
    seen = cspace_convert(rgb, DEUTERANOPIA, "sRGB1")
    return np.clip(seen, 0, 1)

def closest_pair(palette):
    # Smallest perceptual distance between two chain colors,
    # as a deuteranope sees them (CAM02-UCS color space)
    ucs = {c: cspace_convert(seen_by_deuteranope(v), "sRGB1",
                             "CAM02-UCS")
           for c, v in palette.items() if c != "Other"}
    return min((np.linalg.norm(ucs[a] - ucs[b]), a, b)
               for a, b in combinations(ucs, 2))

palettes = [("default", DEFAULT), ("Okabe-Ito", CHAIN)]
for name, palette in palettes:
    distance, a, b = closest_pair(palette)
    print(f"{name:10} closest: {a} / {b}, {distance:.1f}")

In [ ]:
CHAIN_ORDER = list(CHAIN)
region_shares = (pd.crosstab(minisupers.region, minisupers.chain, normalize="index")
                 .mul(100).loc[REGIONS[::-1], CHAIN_ORDER])
as_deuteranope = lambda color: tuple(seen_by_deuteranope(color))

def stacked_bars(ax, palette, recolor=lambda c: c):
    # 100% stacked bars of chain shares by region
    left = np.zeros(len(region_shares))
    for chain in CHAIN_ORDER:
        values = region_shares[chain].values
        ax.barh(range(4), values, left=left, color=recolor(palette[chain]), edgecolor="white", lw=0.5, height=0.7)
        left += values
    ax.set_yticks(range(4), region_shares.index); ax.tick_params(axis="y", length=0); ax.set_xticks([])
    ax.spines[["left", "bottom"]].set_visible(False)

fig, axes = plt.subplots(2, 2, figsize=(WIDTH, 3.3), sharex=True, sharey=True)
for j, (name, palette) in enumerate([("Matplotlib default", DEFAULT), ("Okabe–Ito", CHAIN)]):
    for i, (view, recolor) in enumerate([("as drawn", lambda c: c), ("simulated deuteranopia", as_deuteranope)]):
        stacked_bars(axes[i, j], palette, recolor)
        axes[i, j].set_title(f"{name}, {view}", loc="left", pad=3)
    handles = [plt.Rectangle((0, 0), 1, 1, color=palette[c]) for c in CHAIN_ORDER]
    axes[1, j].legend(handles, CHAIN_ORDER, ncol=4, loc="upper left", bbox_to_anchor=(-0.02, -0.02),
                      frameon=False, fontsize=6, handlelength=0.9, columnspacing=0.6, handletextpad=0.3)
fig.subplots_adjust(left=0.13, right=0.99, top=0.94, bottom=0.17, wspace=0.3, hspace=0.35)
save(fig, "fig12_cvd_simulation")

## 12.5 Comparing Categories

In [ ]:
# Each chain's share of the minisupers in each region
REGIONS = ["Norte", "Centro Norte", "Centro", "Sur"]
# normalize="index": each region's shares sum to 1
shares = (pd.crosstab(minisupers["region"],
                      minisupers["chain"], normalize="index")
          .mul(100).loc[REGIONS, list(CHAIN)])
print(shares.T.round(1).to_string())     # chains as rows

In [ ]:
# A 100% stacked bar: a bar per region, a segment per chain
rows = shares.loc[REGIONS[::-1]]          # Norte at the top
fig, ax = plt.subplots(figsize=(6, 2.3))
left = np.zeros(len(rows))
for chain, color in CHAIN.items():
    values = rows[chain].values
    ax.barh(rows.index, values, left=left, color=color,
            edgecolor="white", label=chain)
    for y, (x0, w) in enumerate(zip(left, values)):
        if w >= 6:                  # label wide segments
            ax.text(x0 + w / 2, y, f"{w:.0f}%",
                    ha="center", va="center")
    left += values
ax.set_xlim(0, 100)

In [ ]:
fig, ax = plt.subplots(figsize=(WIDTH, 2.3))
left = np.zeros(4)
for chain in CHAIN_ORDER:
    values = region_shares[chain].values
    ax.barh(range(4), values, left=left, color=CHAIN[chain], edgecolor="white", lw=0.8, height=0.68)
    for i, (start, width) in enumerate(zip(left, values)):     # label segments wide enough to hold text
        if width >= 6:
            dark_text = chain in ("Other", "Neto", "BA Express")
            ax.text(start + width / 2, i, f"{width:.0f}%", ha="center", va="center", fontsize=6.5,
                    color="black" if dark_text else "white")
    left += values
counts = minisupers.region.value_counts()
ax.set_yticks(range(4), [f"{r}\n({counts[r]:,} stores)" for r in region_shares.index])
ax.set_xlim(0, 100); ax.set_xticks([0, 25, 50, 75, 100], ["0%", "25%", "50%", "75%", "100%"])
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
ax.legend([plt.Rectangle((0, 0), 1, 1, color=CHAIN[c]) for c in CHAIN_ORDER], CHAIN_ORDER, ncol=7,
          loc="lower center", bbox_to_anchor=(0.45, 1.0), frameon=False, handlelength=1, columnspacing=1.0)
save(fig, "fig12_stacked_chains")

In [ ]:
# Slope chart: a line from each state's rank by count (left)
# to its rank by rate (right); steep lines are big moves
fig, ax = plt.subplots(figsize=(6, 6.6))
for _, s in oxxo_state.iterrows():
    ax.plot([0, 1], [s["rank_count"], s["rank_rate"]],
            color="#C9C9C9", marker="o", ms=2)
ax.set_ylim(32.8, 0.2)                    # rank 1 at the top
ax.axis("off")

In [ ]:
# Slope chart: each state's rank by store count against its rank per capita
ranks = oxxo_state.assign(r_count=oxxo_state.stores.rank(ascending=False, method="first"),
                          r_capita=oxxo_state.per10k.rank(ascending=False, method="first"))
featured = {"Estado de México": "#D55E00", "Baja California Sur": "#0072B2", "Nuevo León": "#000000",
            "Sonora": "#009E73", "Ciudad de México": "#CC79A7"}
fig, ax = plt.subplots(figsize=(WIDTH, 6.6))
for _, row in ranks.iterrows():
    color = featured.get(row["name"], LIGHT); is_featured = row["name"] in featured
    ax.plot([0, 1], [row.r_count, row.r_capita], color=color, lw=1.8 if is_featured else 0.8,
            marker="o", ms=3.5 if is_featured else 2, zorder=3 if is_featured else 1)
    style = dict(fontsize=6.5, va="center", color=color if is_featured else "#777777",
                 fontweight="bold" if is_featured else "normal")
    ax.text(-0.04, row.r_count, f"{row['name']}  {int(row.stores):,}", ha="right", **style)
    ax.text(1.04, row.r_capita, f"{row.per10k:.2f}  {row['name']}", ha="left", **style)
ax.set(ylim=(32.8, 0.2), xlim=(-0.05, 1.05)); ax.axis("off")
ax.text(0, -0.6, "Rank by store count", ha="center", fontsize=7.5)
ax.text(1, -0.6, "Rank per 10,000 residents", ha="center", fontsize=7.5)
fig.subplots_adjust(left=0.3, right=0.7, top=0.95, bottom=0.02)
save(fig, "fig12_slope_ranks")

In [ ]:
from scipy.spatial import cKDTree

# Store coordinates in meters (EPSG 6372) for distances
lonlat = gpd.points_from_xy(minisupers["longitud"],
                            minisupers["latitud"])
points = gpd.GeoDataFrame(minisupers, geometry=lonlat,
                          crs=4326).to_crs(6372)
minisupers["x"] = points.geometry.x
minisupers["y"] = points.geometry.y
population = oxxo_mun.set_index("key")["pop"]

profiles = {}
for chain in list(CHAIN)[:-1]:            # all but "Other"
    s = minisupers[minisupers["chain"] == chain]
    # k=2: the nearest point to each store is the store itself
    xy = s[["x", "y"]]
    nearest, _ = cKDTree(xy).query(xy, k=2)
    profiles[chain] = {
        "Norte": 100 * (s["region"] == "Norte").mean(),
        "Sur": 100 * (s["region"] == "Sur").mean(),
        "Big": 100 * (s["key"].map(population) >= 1e6).mean(),
        "States": 100 * s["cve_ent"].nunique() / 32,
        "Spacing": np.median(nearest[:, 1])}
profiles = pd.DataFrame(profiles).T
print(profiles.round(1).to_string())

In [ ]:
# The radar chart: every axis scaled to its own maximum, metres beside percentages
metrics = ["Norte", "Sur", "Big", "States", "Spacing"]
axis_labels = ["Share in Norte", "Share in Sur", "Share in million-plus\nmunicipalities", "States covered",
               "Median spacing\nto own chain"]
scaled = profiles[metrics] / profiles[metrics].max()
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist(); angles += angles[:1]
fig = plt.figure(figsize=(3.8, 3.6)); ax = fig.add_subplot(polar=True)
for chain in profiles.index:
    values = scaled.loc[chain].tolist(); values += values[:1]
    ax.plot(angles, values, color=CHAIN[chain], lw=1.2, label=chain)
    ax.fill(angles, values, color=CHAIN[chain], alpha=0.08)
ax.set_xticks(angles[:-1], axis_labels, fontsize=6); ax.tick_params(axis="x", pad=10)
ax.set_yticklabels([]); ax.set_ylim(0, 1.05)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.12), frameon=False, fontsize=6)
save(fig, "fig12_radar_before")

In [ ]:
# One small panel per measure; the shares use a 0-100 axis
titles = {"Norte": "% in Norte", "Sur": "% in Sur",
          "Big": "% in million-plus\nmunicipalities",
          "States": "% of 32 states",
          "Spacing": "Median spacing\nto own chain (m)"}
y = np.arange(len(profiles))[::-1]
fig, axes = plt.subplots(1, 5, figsize=(6, 2.1), sharey=True)
for ax, (measure, title) in zip(axes, titles.items()):
    values = profiles[measure].values
    ax.hlines(y, 0, values, color="#D9D9D9")
    ax.scatter(values, y, s=18, zorder=3,
               color=[CHAIN[c] for c in profiles.index])
    ax.set_title(title, loc="left")
    if measure != "Spacing":
        ax.set_xlim(-4, 105)
axes[0].set_yticks(y, profiles.index)

In [ ]:
# The same profiles as small-multiple dot plots: shares on a shared 0-100 axis, spacing in metres
titles = ["% in Norte", "% in Sur", "% in million-plus\nmunicipalities", "% of 32 states", "Median spacing\nto own chain (m)"]
y = np.arange(len(profiles))[::-1]
fig, axes = plt.subplots(1, 5, figsize=(WIDTH, 2.1), sharey=True, gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.1]})
for ax, metric, title in zip(axes, metrics, titles):
    values = profiles[metric].values
    ax.hlines(y, 0, values, color=LIGHT, lw=1)
    ax.scatter(values, y, color=[CHAIN[c] for c in profiles.index], s=18, zorder=3, clip_on=False)
    ax.set_title(title, fontsize=6.5, loc="left")
    ax.grid(axis="x", color="#EEEEEE", lw=0.6); ax.set_axisbelow(True)
    if metric == "Spacing":
        ax.set_xlim(-40, 1200); ax.set_xticks([0, 400, 800, 1200])
    else:
        ax.set_xlim(-4, 105)
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
axes[0].set_yticks(y, profiles.index)
for label, chain in zip(axes[0].get_yticklabels(), profiles.index):
    label.set_color(CHAIN[chain])
fig.tight_layout()
save(fig, "fig12_dotplot_after")

In [ ]:
# DENUE's staff band: share of stores reporting 0-5 people
small = minisupers["per_ocu"].str.startswith("0 a 5")
print((100 * small.groupby(minisupers["chain"]).mean())
      .round(1).loc[list(CHAIN)].to_string())

## 12.6 Distributions and Density

In [ ]:
# Distance from each OXXO to the nearest store of a rival,
# under two definitions of who counts as a rival
oxxo = minisupers[minisupers["chain"] == "OXXO"]
rival_sets = {
    "7-Eleven, Circle K": ["7-Eleven", "Circle K"],
    "all five": ["7-Eleven", "Circle K", "3B", "Neto",
                 "BA Express"]}
km = {}
for label, rivals in rival_sets.items():
    is_rival = minisupers["chain"].isin(rivals)
    tree = cKDTree(minisupers.loc[is_rival, ["x", "y"]])
    meters, _ = tree.query(oxxo[["x", "y"]])
    km[label] = pd.Series(meters / 1000, index=oxxo.index)

medians = pd.DataFrame(
    {label: d.groupby(oxxo["region"]).median()
     for label, d in km.items()})
print("Median distance to the nearest rival (km)")
print(medians.loc[REGIONS].round(2).to_string())

In [ ]:
# One ECDF per region: the share of OXXO stores whose
# nearest rival is within each distance
REGION_COLOR = {"Norte": "#132B45", "Centro Norte": "#3B6E9F",
                "Centro": "#7FA8D0", "Sur": "#B7CFE6"}
fig, axes = plt.subplots(1, 2, figsize=(6, 2.4), sharey=True)
for ax, (label, d) in zip(axes, km.items()):
    for region in REGIONS:
        in_region = d[oxxo["region"] == region]
        sns.ecdfplot(in_region.clip(lower=0.01), ax=ax,
                     color=REGION_COLOR[region], label=region)
    ax.set_xscale("log")       # distances span 4 decades
    ax.set_title(f"Rivals: {label}", loc="left")
axes[1].legend(frameon=False)

# Norte stores far from the nearest 7-Eleven or Circle K
north = km["7-Eleven, Circle K"][oxxo["region"] == "Norte"]
for limit in [10, 100]:
    share = 100 * (north > limit).mean()
    print(f"Beyond {limit} km: {share:.0f}%")

In [ ]:
oxxo = minisupers[minisupers.chain == "OXXO"]
distance_km = {"(a)  Rivals: 7-Eleven and Circle K": km["7-Eleven, Circle K"].clip(lower=0.01),
               "(b)  Rivals: all five named chains": km["all five"].clip(lower=0.01)}

fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.4), sharey=True)
for ax, (label, km) in zip(axes, distance_km.items()):
    for region in REGIONS:
        sns.ecdfplot(km[oxxo.region == region], ax=ax, color=REGION_COLOR[region], lw=1.5, label=region)
    ax.axhline(0.5, color=GRAY, lw=0.5, ls=":")
    ax.set(xscale="log", xlim=(0.05, 600), xlabel="Distance to the nearest rival (log scale)")
    ax.set_xticks([0.1, 1, 10, 100], ["100 m", "1 km", "10 km", "100 km"])
    ax.text(0.02, 1.03, label, transform=ax.transAxes, fontsize=7)
axes[0].set_ylabel("Share of OXXO stores within distance")
axes[0].set_yticks([0, 0.25, 0.5, 0.75, 1], ["0%", "25%", "50%", "75%", "100%"])
axes[1].legend(frameon=False, loc="lower right")
fig.tight_layout()
save(fig, "fig12_ecdf_rivals")

In [ ]:
# Hexagonal bins over metropolitan Monterrey: the 13
# municipalities of the metro area (state code 19)
MONTERREY = ["19" + m for m in ["006", "009", "010", "018",
             "019", "021", "026", "031", "039", "045", "046",
             "048", "049"]]
metro = minisupers[minisupers["key"].isin(MONTERREY)]
# A window centered on the median OXXO, 39 by 34 km
is_oxxo = metro["chain"] == "OXXO"
cx, cy = metro.loc[is_oxxo, ["x", "y"]].median()
extent = (cx - 19_550, cx + 19_550,      # x from, x to
          cy - 17_000, cy + 17_000)      # y from, y to
in_window = (metro["x"].between(*extent[:2])
             & metro["y"].between(*extent[2:]))

fig, axes = plt.subplots(1, 2, figsize=(6, 2.9))
for ax, chain in zip(axes, ["OXXO", "7-Eleven"]):
    s = metro[in_window & (metro["chain"] == chain)]
    # 22 hexagons across; empty ones are not drawn (mincnt)
    hexes = ax.hexbin(s["x"], s["y"], gridsize=22,
                      extent=extent, cmap="cividis",
                      mincnt=1, vmin=1, vmax=20)
    ax.set_aspect("equal")
    ax.set_axis_off()
    most = hexes.get_array().max()
    print(f"{chain}: {len(s):,} stores, up to {most:.0f} "
          f"in one hexagon")

In [ ]:
munis_m, states_m = munis.to_crs(6372), states.to_crs(6372)

# Hexagonal bins over metropolitan Monterrey: OXXO (a) and 7-Eleven (b)
MONTERREY = ["19" + m for m in ["006", "009", "010", "018", "019", "021", "026", "031", "039", "045", "046", "048", "049"]]
metro = minisupers[minisupers.key.isin(MONTERREY)]
cx, cy = metro.loc[metro.chain == "OXXO", ["x", "y"]].median()
R = 17_000                                                   # half-height of the view, in metres
extent = (cx - 1.15 * R, cx + 1.15 * R, cy - R, cy + R)
metro = metro[metro.x.between(extent[0], extent[1]) & metro.y.between(extent[2], extent[3])]
metro_lines = munis_m[munis_m.key.isin(MONTERREY)].boundary

fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.9))
for ax, chain, tag in [(axes[0], "OXXO", "(a) OXXO"), (axes[1], "7-Eleven", "(b) 7-Eleven")]:
    s = metro[metro.chain == chain]
    hexes = ax.hexbin(s.x, s.y, gridsize=22, extent=extent, cmap="cividis", mincnt=1, vmin=1, vmax=20,
                      linewidths=0.2, edgecolors="white")
    metro_lines.plot(ax=ax, color=GRAY, lw=0.4)
    ax.set_xlim(extent[:2]); ax.set_ylim(extent[2:]); ax.set_aspect("equal"); ax.set_axis_off()
    ax.text(0.01, 1.02, f"{tag}  ({len(s):,} stores)", transform=ax.transAxes, fontsize=7)
axes[0].plot([extent[0] + 2000, extent[0] + 12000], [extent[2] + 2000] * 2, color="black", lw=1.5)
axes[0].text(extent[0] + 7000, extent[2] + 2800, "10 km", ha="center", fontsize=6)
cax = fig.add_axes([0.35, 0.1, 0.3, 0.03])
bar = fig.colorbar(hexes, cax=cax, orientation="horizontal", extend="max", ticks=[1, 5, 10, 15, 20])
bar.set_label("Stores per hexagon (about 1.8 km across)"); bar.outline.set_linewidth(0.3)
fig.subplots_adjust(left=0.02, right=0.98, top=0.94, bottom=0.24, wspace=0.05)
save(fig, "fig12_hexbin_monterrey")

## 12.7 Maps beyond the Choropleth

In [ ]:
# One circle per municipality, its area proportional to its
# OXXO stores; matplotlib's `s` is already an area
symbols = (munis[["key", "geometry"]]
           .merge(oxxo_mun, on="key").to_crs(6372))
symbols["geometry"] = symbols.representative_point()
# Drop empty municipalities; draw big circles first
symbols = (symbols[symbols["stores"] > 0]
           .sort_values("stores", ascending=False))

fig, ax = plt.subplots(figsize=(6, 3.9))
states.to_crs(6372).plot(ax=ax, color="#F2F2F2",
                         edgecolor="white")
ax.scatter(symbols.geometry.x, symbols.geometry.y,
           s=0.4 * symbols["stores"], color="#4682B4",
           alpha=0.55, edgecolor="white", linewidth=0.2)
ax.set_axis_off()

top = symbols.iloc[0]
print(f"{len(symbols):,} municipalities with an OXXO")
print(f"Largest: {top['NOM_MUN']}, {top['stores']:,} stores")

In [ ]:
# Proportional symbols: circle area proportional to the number of OXXO stores in each municipality
points = munis_m[["key", "geometry"]].merge(oxxo_mun, on="key")
points["geometry"] = points.geometry.representative_point()
points = points[points.stores > 0].sort_values("stores", ascending=False)    # big circles drawn first
AREA_PER_STORE = 0.4                                                         # points^2 per store

fig, ax = plt.subplots(figsize=(WIDTH, 3.9))
states_m.plot(ax=ax, color="#F2F2F2", edgecolor="white", lw=0.4)
ax.scatter(points.geometry.x, points.geometry.y, s=points.stores * AREA_PER_STORE, color=STEEL, alpha=0.55,
           edgecolor="white", lw=0.2)
x0, y0, x1, y1 = states_m.total_bounds
ax.set_xlim(x0 - 0.03 * (x1 - x0), x1 + 0.01 * (x1 - x0)); ax.set_ylim(y0 - 0.02 * (y1 - y0), y1 + 0.06 * (y1 - y0))
ax.set_axis_off()
for n in [10, 100, 500]:                                                     # legend circles
    ax.scatter([], [], s=n * AREA_PER_STORE, color=STEEL, alpha=0.55, edgecolor="white", lw=0.2, label=str(n))
ax.legend(title="OXXO stores\nper municipality", loc="lower left", bbox_to_anchor=(0.02, 0.06), frameon=False,
          labelspacing=1.8, borderpad=1.0, fontsize=6.5, title_fontsize=6.5)
save(fig, "fig12_propsymbol")

In [ ]:
# Monterrey's OXXO stores within 500 m of a 7-Eleven
RADIUS = 500
metro = minisupers[minisupers["key"].isin(MONTERREY)]
seven = metro[metro["chain"] == "7-Eleven"]
oxxo_mty = metro[metro["chain"] == "OXXO"].copy()
tree = cKDTree(seven[["x", "y"]])
nearest, _ = tree.query(oxxo_mty[["x", "y"]])
oxxo_mty["inside"] = nearest <= RADIUS

inside = oxxo_mty["inside"]
print(f"Metro: {inside.sum():,} of {len(inside):,} "
      f"({100 * inside.mean():.0f}%)")
by_mun = (oxxo_mty.groupby("key")["inside"]
          .agg(["sum", "size"]))
by_mun["pct"] = 100 * by_mun["sum"] / by_mun["size"]
names = munis.set_index("key")["NOM_MUN"]
table = (by_mun.rename(index=names).rename_axis(None)
         .sort_values("size", ascending=False))
print(table.round(0).astype(int).to_string())

In [ ]:
# Circles of 500 m around each 7-Eleven, merged into one shape
centers = gpd.points_from_xy(seven["x"], seven["y"])
circles = (gpd.GeoSeries(centers, crs=6372)
           .buffer(RADIUS).union_all())

fig, ax = plt.subplots(figsize=(6, 4.1))
gpd.GeoSeries([circles]).plot(ax=ax, color="#009E73",
                              alpha=0.14)
ax.scatter(seven["x"], seven["y"], s=5, color="#009E73")
# OXXO inside a circle: filled; outside: hollow
for is_in, face in [(True, "#D55E00"), (False, "white")]:
    s = oxxo_mty[oxxo_mty["inside"] == is_in]
    ax.scatter(s["x"], s["y"], s=6, facecolor=face,
               edgecolor="#D55E00", linewidth=0.7)
ax.set_aspect("equal")

In [ ]:
from shapely.geometry import box

# The window of the printed figure: centered 2 km south-west of the median 7-Eleven
cx, cy = seven[["x", "y"]].median() - 2000
W, X = 4500, 4500 * 1.35
window = box(cx - X, cy - W, cx + X, cy + W)
in_window = lambda df: df[df.x.between(cx - X, cx + X) & df.y.between(cy - W, cy + W)]
seven_w, oxxo_w = in_window(seven), in_window(oxxo_mty)
print(f"Window: {oxxo_w.inside.sum()} of {len(oxxo_w)} OXXO ({100 * oxxo_w.inside.mean():.0f}%)")

circles = gpd.GeoSeries(gpd.points_from_xy(seven.x, seven.y), crs=6372).buffer(RADIUS).union_all()
visible = munis_m[munis_m.key.isin(MONTERREY) & munis_m.intersects(window)].copy()
visible["clipped"] = visible.intersection(window)
LABELS = {"19039": "Monterrey", "19046": "San Nicolás\nde los Garza", "19026": "Guadalupe", "19019": "San Pedro\nGarza García"}

# Put the Monterrey label where it covers the fewest stores
all_pts = np.r_[seven_w[["x", "y"]].values, oxxo_w[["x", "y"]].values]
grid = np.c_[[g.ravel() for g in np.meshgrid(np.linspace(cx - X, cx + X, 90), np.linspace(cy - W, cy + W, 70))]].T
core = visible.set_index("key").clipped["19039"].buffer(-600)
candidates = grid[gpd.GeoSeries(gpd.points_from_xy(grid[:, 0], grid[:, 1])).within(core).values]
spacing, _ = cKDTree(all_pts).query(candidates, k=6)
label_at = {"19039": candidates[np.argmax(spacing.mean(axis=1))]}

fig, ax = plt.subplots(figsize=(WIDTH, 4.1))
gpd.GeoSeries([circles], crs=6372).plot(ax=ax, facecolor=CHAIN["7-Eleven"], alpha=0.14, edgecolor="none")
visible.boundary.plot(ax=ax, color=GRAY, linewidth=0.5, linestyle="dashed")
ax.scatter(seven_w.x, seven_w.y, s=5, color=CHAIN["7-Eleven"], zorder=3, label="7-Eleven")
ax.scatter(oxxo_w[oxxo_w.inside].x, oxxo_w[oxxo_w.inside].y, s=6, color=CHAIN["OXXO"], zorder=4,
           label="OXXO within 500 m of a 7-Eleven")
ax.scatter(oxxo_w[~oxxo_w.inside].x, oxxo_w[~oxxo_w.inside].y, s=6, facecolor="white", edgecolor=CHAIN["OXXO"],
           lw=0.7, zorder=4, label="OXXO farther than 500 m")
for _, row in visible.iterrows():
    at = label_at.get(row.key, (row.clipped.representative_point().x, row.clipped.representative_point().y))
    ax.text(*at, LABELS[row.key], fontsize=6.5, style="italic", color="#555555", ha="center", va="center", zorder=6,
            path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])
ax.set(xlim=(cx - X, cx + X), ylim=(cy - W, cy + W)); ax.set_aspect("equal"); ax.set_axis_off()
ax.plot([cx + X - 2150, cx + X - 150], [cy - 0.93 * W] * 2, color="black", lw=1.5)
ax.text(cx + X - 1150, cy - 0.88 * W, "2 km", ha="center", fontsize=6, path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])
ax.legend(loc="upper left", bbox_to_anchor=(0, 1.07), ncol=3, frameon=False, handletextpad=0.2, columnspacing=1)
fig.subplots_adjust(left=0, right=1, top=0.94, bottom=0)
save(fig, "fig12_radius_monterrey")

## 12.8 Interactivity

The listings show the core of each interactive figure. After each one, a notebook-only cell
draws the finished version used for the printed figures and for sharing as HTML.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

# A template holds styling that every figure inherits
pio.templates["dsbe"] = go.layout.Template(layout=dict(
    font=dict(family="Helvetica, Arial, sans-serif", size=13),
    colorway=list(CHAIN.values())[:6],
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="#CCCCCC"),
    yaxis=dict(gridcolor="#EFEFEF", linecolor="#CCCCCC"),
    hoverlabel=dict(bgcolor="white", bordercolor="#BBBBBB")))
pio.templates.default = "dsbe"

In [ ]:
# The municipal map over a street basemap, with a hover card
muni_json = json.loads(munis[["key", "geometry"]].to_json())
m = oxxo_mun
card = ("<b>%{customdata[0]}</b>"
        "<br>Population: %{customdata[1]:,}"
        "<br>OXXO stores: %{customdata[2]}"
        "<br>Per 10,000: %{customdata[3]:.2f}<extra></extra>")
hover_map = go.Figure(go.Choroplethmapbox(
    geojson=muni_json, featureidkey="properties.key",
    locations=m["key"], z=m["per10k"], zmin=0, zmax=6,
    colorscale="Cividis", marker_opacity=0.78,
    marker_line_width=0.2, hovertemplate=card,
    customdata=m[["NOM_MUN", "pop", "stores", "per10k"]]))

# Buttons swap the values the single trace shows
floored = m["per10k"].where(m["pop"] >= FLOOR)  # NaN if small
views = {"Per 10,000": m["per10k"],
         "Per 10,000, 20k+": floored}
buttons = [dict(label=label, method="restyle",
                args=[{"z": [values.tolist()]}])
           for label, values in views.items()]
hover_map.update_layout(
    mapbox=dict(style="carto-positron", zoom=4.1,   # no token
                center=dict(lat=23.6, lon=-102.3)),
    updatemenus=[dict(type="buttons", direction="right",
                      buttons=buttons)])
hover_map.show()

In [ ]:
# Finished version: three views (counts, rates, floored rates) and a full hover card
# Municipal map over a street basemap. Three views share one trace; the buttons swap its values.
muni_json = json.loads(munis[["key", "geometry"]].to_json())
m = oxxo_mun.copy()
m["state"] = m.NOM_ENT.map(short)
details = np.c_[m.NOM_MUN, m.state, m["pop"].map("{:,.0f}".format), m.stores.astype(int), m.per10k.round(2)]

cividis = px.colors.get_colorscale("Cividis")
grey_then_cividis = [[0, "#D9D9D9"], [1 / 7 - 1e-6, "#D9D9D9"]] + [[1 / 7 + 6 / 7 * s, c] for s, c in cividis]
views = [  # label, values, color range, color scale, colorbar ticks
    ("Store count", np.log10(m.stores.clip(lower=0.5)), (np.log10(0.5), np.log10(700)), cividis,
     "Stores", np.log10([1, 10, 100, 500]), ["1", "10", "100", "500"]),
    ("Per 10,000 residents", m.per10k, (0, 6), cividis, "Per 10k", [0, 2, 4, 6], ["0", "2", "4", "6+"]),
    ("Per 10,000, population ≥ 20,000", np.where(m["pop"] >= FLOOR, m.per10k, -1), (-1, 6), grey_then_cividis,
     "Per 10k", [-0.5, 0, 2, 4, 6], ["under 20k", "0", "2", "4", "6+"]),
]
label, z, (zmin, zmax), scale, title, ticks, ticktext = views[1]
hover_map = go.Figure(go.Choroplethmapbox(
    geojson=muni_json, featureidkey="properties.key", locations=m.key, z=z, zmin=zmin, zmax=zmax,
    colorscale=scale, marker_opacity=0.78, marker_line_width=0.2, marker_line_color="white", customdata=details,
    hovertemplate=("<b>%{customdata[0]}</b>, %{customdata[1]}<br>Population 2020: %{customdata[2]}"
                   "<br>OXXO stores: %{customdata[3]}<br>Per 10,000 residents: %{customdata[4]}<extra></extra>"),
    colorbar=dict(title=title, tickvals=ticks, ticktext=ticktext, thickness=12, len=0.5)))
buttons = [dict(label=label, method="restyle",
                args=[{"z": [np.round(z, 3).tolist()], "zmin": lo, "zmax": hi, "colorscale": [scale],
                       "colorbar.title.text": title, "colorbar.tickvals": [list(ticks)], "colorbar.ticktext": [ticktext]}])
           for label, z, (lo, hi), scale, title, ticks, ticktext in views]
hover_map.update_layout(
    mapbox=dict(style="carto-positron", center=dict(lat=23.6, lon=-102.3), zoom=4.1),   # free basemap, no token
    height=640, margin=dict(l=0, r=0, t=50, b=0),
    updatemenus=[dict(type="buttons", direction="right", x=0.01, y=1.06, xanchor="left", yanchor="bottom",
                      active=1, buttons=buttons)])
hover_map.show(config={"scrollZoom": True})
# hover_map.write_html(FIGURES / "ch12_municipal_map.html", include_plotlyjs="cdn")

In [ ]:
# Print version: the same map without a basemap, with one tooltip drawn in by hand
huasabas = munis.merge(oxxo_mun[["key", "pop", "stores", "per10k"]], on="key").query("NOM_MUN == 'Huásabas'").iloc[0]
spot = huasabas.geometry.representative_point()
printed = go.Figure(go.Choropleth(
    geojson=muni_json, featureidkey="properties.key", locations=m.key, z=m.per10k, zmin=0, zmax=6,
    colorscale="Cividis", marker_line_width=0.1, marker_line_color="white",
    colorbar=dict(title="Per 10k", thickness=12, len=0.55, tickvals=[0, 2, 4, 6], ticktext=["0", "2", "4", "6+"], x=0.95, y=0.55)))
printed.add_trace(go.Scattergeo(lon=[spot.x], lat=[spot.y], mode="markers", hoverinfo="skip",
                                marker=dict(size=12, color="rgba(0,0,0,0)", line=dict(color=CHAIN["OXXO"], width=2.5))))
printed.add_trace(go.Scattergeo(lon=[spot.x - 1.0], lat=[spot.y + 0.05], mode="text", text=["<b>A</b>"],
                                textfont=dict(size=13, color=CHAIN["OXXO"]), hoverinfo="skip"))
printed.add_annotation(
    text=(f"<b>A</b>  Huásabas, Sonora<br>Population 2020: {int(huasabas['pop']):,}<br>"
          f"OXXO stores: {int(huasabas.stores)}<br>Per 10,000 residents: {huasabas.per10k:.2f}"),
    x=0.03, y=0.06, xref="paper", yref="paper", xanchor="left", yanchor="bottom", showarrow=False, align="left",
    bgcolor="white", bordercolor="#999999", borderwidth=1, borderpad=6, font=dict(size=11))
printed.update_layout(
    showlegend=False, width=720, height=470, font=dict(size=12), margin=dict(l=5, r=5, t=55, b=5),
    geo=dict(visible=False, projection_type="mercator", lonaxis_range=[-117.5, -86.5], lataxis_range=[14.3, 33.2],
             domain=dict(x=[0, 0.93], y=[0, 1])),
    updatemenus=[dict(type="buttons", direction="right", x=0, y=1.08, xanchor="left", active=1,
                      buttons=[dict(label=v[0], method="skip") for v in views])])
printed.write_image(FIGURES / "fig12_plotly_municipal_hover.png", scale=4)

In [ ]:
# Chain shares by state; a menu re-sorts the rows
by_state = (pd.crosstab(minisupers["cve_ent"],
                        minisupers["chain"],
                        normalize="index")
            .mul(100)[list(CHAIN)])
codes = oxxo_mun["key"].str[:2]                # state code
names = dict(zip(codes, oxxo_mun["NOM_ENT"]))
by_state.index = by_state.index.map(names)

def ordered(column):
    rows = by_state.sort_values(column, ascending=False)
    return {"z": [rows.clip(upper=30).values.tolist()],
            "y": [rows.index.tolist()],
            "text": [rows.round(0).astype(int)
                     .values.tolist()]}

heat = go.Figure(go.Heatmap(x=list(CHAIN),
                            colorscale="Blues",
                            texttemplate="%{text}",
                            **{k: v[0] for k, v in
                               ordered("OXXO").items()}))
heat.update_yaxes(autorange="reversed")  # first row on top
sort_menu = [dict(label=f"Sort by {c}", method="restyle",
                  args=[ordered(c)]) for c in CHAIN]
heat.update_layout(updatemenus=[dict(buttons=sort_menu)])
heat.show()

In [ ]:
# Finished version: states in OXXO-density order, a steel-blue scale and detailed hover
state_shares = (pd.crosstab(minisupers.cve_ent, minisupers.chain, normalize="index").mul(100)[CHAIN_ORDER]
                .rename(index=dict(zip(oxxo_state.cve_ent, oxxo_state.name))))
store_counts = minisupers.cve_ent.map(dict(zip(oxxo_state.cve_ent, oxxo_state.name))).value_counts()
blues = [[0, "#FFFFFF"], [0.35, "#B7CFE6"], [0.7, STEEL], [1, "#132B45"]]

def heatmap_view(rows):
    # Values (capped at 30% so small chains stay visible), cell text and hover text for one row order
    z = state_shares.loc[rows].values
    text = np.where(z == 0, "", np.where(z < 0.5, "<1", np.round(z).astype(int).astype(str)))
    hover = [[f"<b>{s}</b><br>{c}: {z[i, j]:.1f}% of {store_counts[s]:,} minisupers"
              for j, c in enumerate(CHAIN_ORDER)] for i, s in enumerate(rows)]
    return dict(z=np.minimum(z, 30), y=rows, text=text, customdata=hover)

orders = {"OXXO density": oxxo_state.sort_values("per10k", ascending=False).name.tolist()}
orders.update({f"{c} share": state_shares[c].sort_values(ascending=False).index.tolist() for c in CHAIN_ORDER})

heat = go.Figure(go.Heatmap(x=CHAIN_ORDER, colorscale=blues, zmin=0, zmax=30, texttemplate="%{text}",
                            hovertemplate="%{customdata}<extra></extra>", xgap=2, ygap=2,
                            colorbar=dict(title=dict(text="Share of<br>minisupers<br> ", side="top"), thickness=10,
                                          len=0.35, y=0.82, tickvals=[0, 10, 20, 30], ticktext=["0%", "10%", "20%", "30%+"]),
                            **heatmap_view(orders["OXXO density"])))
sort_buttons = [dict(label=k, method="restyle", args=[{key: [value] for key, value in heatmap_view(rows).items()}])
                for k, rows in orders.items()]
heat.update_yaxes(autorange="reversed", showgrid=False, ticks="", showline=False)
heat.update_xaxes(side="top", showline=False, ticks="")
heat.update_layout(width=820, height=1000, margin=dict(l=150, r=10, t=110, b=10),
                   updatemenus=[dict(buttons=sort_buttons, direction="down", x=0, y=1.06, xanchor="left", yanchor="bottom")],
                   annotations=[dict(text="Sort states by", x=0, y=1.095, xref="paper", yref="paper", xanchor="left",
                                     yanchor="bottom", showarrow=False, font=dict(size=12, color="#666"))])
heat.show()

In [ ]:
# Print version: smaller canvas so the text lands at a readable size on the page
heat_print = go.Figure(heat)
heat_print.layout.updatemenus = ()        # no menu on paper
heat_print.layout.annotations = ()
heat_print.update_layout(width=600, height=720, font_size=11, margin=dict(l=120, r=10, t=40, b=10))
heat_print.update_xaxes(tickangle=0, tickfont_size=10)
heat_print.update_traces(textfont_size=9.5)
heat_print.write_image(FIGURES / "fig12_plotly_heatmap.png", scale=4)

In [ ]:
# Retail revenue by state and month, in long format
emec = pd.read_csv(DATA / "emec_retail_monthly.csv")
emec["date"] = pd.to_datetime(dict(year=emec["ANIO"],
                                   month=emec["MES"], day=1))
retail = (emec.pivot(index="date", columns="ENTIDAD",
                     values="M000W")
          .rolling(12).mean()            # 12-month average
          .loc["2018-12-01":])
long = retail.reset_index().melt(id_vars="date",
                                 var_name="state",
                                 value_name="index")
long["month"] = long["date"].dt.strftime("%b %Y")

# animation_frame turns each month into one frame
animation = px.bar(long, x="index", y="state",
                   orientation="h", animation_frame="month",
                   range_x=[85, 128])
animation.add_vline(x=100, line_dash="dot")
animation.show()

In [ ]:
# Finished version: short names, bars colored by region, ordered by the latest value
# One row per state and month, in the long format plotly express expects
region_of = {code: region for region, codes in {
    "Norte": [2, 5, 8, 19, 26, 28], "Centro Norte": [1, 3, 6, 10, 14, 16, 18, 24, 25, 32],
    "Centro": [9, 11, 13, 15, 17, 21, 22, 29], "Sur": [4, 7, 12, 20, 23, 27, 30, 31]}.items() for code in codes}
state_region = {row.NOM_ENT: region_of[int(row.cve_ent)] for row in oxxo_state.itertuples()}
retail_long = retail.reset_index().melt(id_vars="date", var_name="NOM_ENT", value_name="index")
retail_long["state"] = retail_long.NOM_ENT.map(short)
retail_long["region"] = retail_long.NOM_ENT.map(state_region)
retail_long["month"] = retail_long.date.dt.strftime("%b %Y")
latest_order = retail.iloc[-1].sort_values(ascending=False).index.map(short).tolist()

animation = px.bar(retail_long, x="index", y="state", color="region", orientation="h", animation_frame="month",
                   category_orders={"state": latest_order, "region": REGIONS}, color_discrete_map=REGION_COLOR,
                   range_x=[85, 128], hover_data={"index": ":.1f", "month": False},
                   labels={"index": "Real retail revenue, 12-month average (2018 = 100)", "state": "", "region": ""})
animation.add_vline(x=100, line_width=1, line_dash="dot", line_color="#888")
animation.update_layout(width=820, height=900, bargap=0.25, legend=dict(orientation="h", y=1.04, x=0),
                        margin=dict(l=140, r=20, t=60, b=10))
animation.update_yaxes(showgrid=False, ticks="", showline=False)
animation.layout.sliders[0].currentvalue = dict(prefix="", font=dict(size=14))
animation.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 90       # milliseconds per month
animation.show()

In [ ]:
# A card: the selected state's value against a reference
card = go.Indicator(
    mode="number+delta", value=3.46,              # Nuevo Leon
    number=dict(valueformat=".2f"),
    delta=dict(reference=national, valueformat=".2f"),
    title=dict(text="OXXO per 10,000 residents"),
    domain=dict(x=[0.02, 0.31], y=[0.8, 0.97]))

# A picker option sends new values to several traces at once:
# one list entry per trace, in the order of the last argument.
# plotly ignores attributes a trace does not have.
def option(label, cve, shares, series, rate):
    return dict(label=label, method="update", args=[
        {"locations": [[cve], 0, 0, 0],      # map outline
         "x": [0, shares, series.index, 0],   # bars, line
         "y": [0, list(CHAIN), series.values, 0],
         "value": [0, 0, 0, rate]},           # the card
        {}, [1, 2, 4, 5]])

In [ ]:
from plotly.subplots import make_subplots

growth = 100 * (retail.iloc[-1] / retail.loc["2019-12-01"] - 1)    # real growth since Dec 2019

state_json = json.loads(states.to_json())
shares_by_code = pd.crosstab(minisupers.cve_ent, minisupers.chain, normalize="index").mul(100)[CHAIN_ORDER]
minis_by_code = minisupers.cve_ent.value_counts()
oxxo_state["rank"] = oxxo_state.per10k.rank(ascending=False).astype(int)
national_share = 100 * (minisupers.chain == "OXXO").mean()
median_growth, median_state = growth.median(), retail.median(axis=1)
dates = retail.index.strftime("%Y-%m-%d").tolist()
pct_labels = lambda v: [f"{x:.0f}%" if x >= 0.5 else "" for x in v]

def headline(s):
    # The text above the cards and the two panel captions, for state row `s`
    note = dict(xref="paper", yref="paper", xanchor="left", showarrow=False)
    return [dict(text=f"<b>{s['name']}</b>  ·  rank {s['rank']} of 32 by OXXO density", x=0, y=1.075, font=dict(size=17), **note),
            dict(text="Minisuper share by chain", x=0.6, y=0.755, font=dict(size=12, color="#666"), **note),
            dict(text="Real retail revenue, 12-month average: state vs median state (dotted)", x=0.6, y=0.36,
                 font=dict(size=12, color="#666"), **note)]

start = oxxo_state.set_index("cve_ent").loc["19"]          # open on Nuevo León
start["cve_ent"] = "19"
dash = make_subplots(rows=2, cols=2, specs=[[{"type": "geo", "rowspan": 2}, {"type": "xy"}], [None, {"type": "xy"}]])
# trace 0: map of all states; trace 1: outline of the selected state
dash.add_trace(go.Choropleth(geojson=state_json, featureidkey="properties.CVE_ENT", locations=oxxo_state.cve_ent,
    z=oxxo_state.per10k, zmin=0.5, zmax=4.2, colorscale="Cividis", marker_line_color="white", marker_line_width=0.6,
    customdata=np.c_[oxxo_state.name, oxxo_state["rank"]],
    hovertemplate="<b>%{customdata[0]}</b><br>%{z:.2f} OXXO per 10k<br>rank %{customdata[1]} of 32<extra></extra>",
    colorbar=dict(title="OXXO per 10k", orientation="h", x=0.28, y=-0.02, len=0.42, thickness=9)), row=1, col=1)
dash.add_trace(go.Choropleth(geojson=state_json, featureidkey="properties.CVE_ENT", locations=["19"], z=[1],
    showscale=False, hoverinfo="skip", colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
    marker_line_color=CHAIN["OXXO"], marker_line_width=3), row=1, col=1)
# trace 2: chain composition
dash.add_trace(go.Bar(x=shares_by_code.loc["19"], y=CHAIN_ORDER, orientation="h", marker_color=list(CHAIN.values()),
    text=pct_labels(shares_by_code.loc["19"]), textposition="outside", cliponaxis=False,
    hovertemplate="%{y}: %{x:.1f}%<extra></extra>"), row=1, col=2)
# traces 3-4: median state and the selected state
dash.add_trace(go.Scatter(x=retail.index, y=median_state, line=dict(color="#A8A8A8", dash="dot", width=1.6),
    hovertemplate="Median state %{y:.1f}<extra></extra>"), row=2, col=2)
dash.add_trace(go.Scatter(x=retail.index, y=retail[start.NOM_ENT], line=dict(color=STEEL, width=2.8), fill="tonexty",
    fillcolor="rgba(70,130,180,0.08)", hovertemplate="%{x|%b %Y}: %{y:.1f}<extra></extra>"), row=2, col=2)
# traces 5-7: indicator cards, each against a national reference
cards = [("OXXO per 10,000 residents", start.per10k, NATIONAL, ".2f", "", "", f"national {NATIONAL:.2f}"),
         ("OXXO share of minisupers", shares_by_code.loc["19", "OXXO"], national_share, ".1f", "%", " pp",
          f"national {national_share:.1f}%"),
         ("Real retail growth since Dec 2019", growth[start.NOM_ENT], median_growth, ".1f", "%", " pp",
          f"median state {median_growth:.1f}%")]
for k, (title, value, reference, fmt, suffix, delta_suffix, versus) in enumerate(cards):
    dash.add_trace(go.Indicator(mode="number+delta", value=value,
        number=dict(valueformat=fmt, suffix=suffix, font=dict(size=30)),
        delta=dict(reference=reference, valueformat=fmt, suffix=delta_suffix, font=dict(size=14),
                   increasing=dict(color=STEEL), decreasing=dict(color=CHAIN["OXXO"])),
        title=dict(text=f"{title}<br><span style='font-size:11px;color:#888'>vs {versus}</span>", font=dict(size=13)),
        domain=dict(x=[0.02 + 0.33 * k, 0.31 + 0.33 * k], y=[0.8, 0.97])))
# The state picker: each option sends new values to traces 1, 2, 4, 5, 6 and 7.
# Each attribute gets one entry per trace; plotly ignores attributes a trace does not have.
options = []
for _, s in oxxo_state.sort_values("name").iterrows():
    shares = shares_by_code.loc[s.cve_ent]
    options.append(dict(label=s["name"], method="update", args=[{
        "locations": [[s.cve_ent], CHAIN_ORDER, dates, 0, 0, 0],
        "x": [[s.cve_ent], shares.tolist(), dates, 0, 0, 0],
        "y": [[1], CHAIN_ORDER, retail[s.NOM_ENT].round(2).tolist(), 0, 0, 0],
        "text": [[""], pct_labels(shares), [""] * len(dates), 0, 0, 0],
        "value": [0, 0, 0, round(s.per10k, 3), round(shares["OXXO"], 2), round(growth[s.NOM_ENT], 2)]},
        {"annotations": headline(s)}, [1, 2, 4, 5, 6, 7]]))
dash.update_layout(
    geo=dict(domain=dict(x=[0, 0.54], y=[0.03, 0.74]), visible=False, projection_type="mercator", fitbounds="locations"),
    xaxis=dict(domain=[0.6, 1], range=[0, 85], showticklabels=False, ticks="", showline=False),
    yaxis=dict(domain=[0.44, 0.72], autorange="reversed", showgrid=False),
    xaxis2=dict(domain=[0.6, 1]), yaxis2=dict(domain=[0.03, 0.33], title=dict(text="Real retail revenue, 2018 = 100", font=dict(size=11))),
    showlegend=False, width=1200, height=760, margin=dict(l=15, r=20, t=70, b=30), annotations=headline(start),
    updatemenus=[dict(buttons=options, direction="down", x=1.0, y=1.1, xanchor="right",
                      active=[o["label"] for o in options].index("Nuevo León"))])
dash.show()
# dash.write_html(FIGURES / "ch12_dashboard.html", include_plotlyjs="cdn")   # a file you can email

In [ ]:
# Print version: narrower canvas and smaller type, so text is readable at the book's width
dash_print = go.Figure(dash)
dash_print.update_layout(width=800, height=500, font_size=10.5, margin=dict(l=5, r=10, t=70, b=30))
for note in dash_print.layout.annotations:
    note.font.size = 13 if note.font.size == 17 else 10.5
dash_print.update_traces(textfont_size=10, selector=dict(type="bar"))
dash_print.update_traces(number_font_size=22, title_font_size=11, selector=dict(type="indicator"))
dash_print.write_image(FIGURES / "fig12_plotly_dashboard.png", scale=4)

## 12.9 Small Multiples, Sparklines and the Tile Map

In [ ]:
# Small multiples: one map per chain, same frame and scale
CHAINS = list(CHAIN)[:-1]                # leave out "Other"
x0, y0, x1, y1 = states.to_crs(6372).total_bounds
fig, axes = plt.subplots(2, 3, figsize=(6, 3.1))
for ax, chain in zip(axes.flat, CHAINS):
    s = minisupers[minisupers["chain"] == chain]
    states.to_crs(6372).plot(ax=ax, color="#EDEDED",
                             edgecolor="white", linewidth=0.3)
    ax.scatter(s["x"], s["y"], s=1, color=CHAIN[chain],
               linewidth=0)
    ax.set_xlim(x0, x1)                  # every panel shows
    ax.set_ylim(y0, y1)                  # the same country
    ax.set_title(f"{chain}  ({len(s):,})", loc="left")
    ax.set_axis_off()

In [ ]:
states_m = states.to_crs(6372)
CHAIN_ORDER = list(CHAIN)

fig, axes = plt.subplots(2, 3, figsize=(WIDTH, 3.1))
x0, y0, x1, y1 = states_m.total_bounds
state_shapes = states_m.copy(); state_shapes["geometry"] = state_shapes.geometry.simplify(2000)
for ax, chain in zip(axes.flat, CHAIN_ORDER[:-1]):
    s = minisupers[minisupers.chain == chain]
    state_shapes.plot(ax=ax, color="#EDEDED", edgecolor="white", lw=0.3)
    is_oxxo = chain == "OXXO"
    ax.scatter(s.x, s.y, s=0.6 if is_oxxo else 1.2, color=CHAIN[chain], lw=0, alpha=0.6 if is_oxxo else 0.8)
    ax.set(xlim=(x0, x1), ylim=(y0, y1)); ax.set_aspect("equal"); ax.set_axis_off()
    ax.text(0.62, 0.9, f"{chain}\n{len(s):,} stores", transform=ax.transAxes, fontsize=7, fontweight="bold", va="top",
            color="#2C8FC9" if chain == "BA Express" else CHAIN[chain])
fig.subplots_adjust(left=0.005, right=0.995, top=0.99, bottom=0.01, wspace=0.02, hspace=0.02)
save(fig, "fig12_smallmult_chains")

In [ ]:
# A sparkline per state, all on one vertical scale
order = oxxo_state.sort_values("per10k", ascending=False)
low, high = retail.values.min(), retail.values.max()
fig, axes = plt.subplots(16, 2, figsize=(2.2, 3.2))
for ax, state in zip(axes.T.flat, order["NOM_ENT"]):
    series = retail[state]
    ax.plot(series.index, series.values, color="#9A9A9A",
            linewidth=0.7)
    ax.plot(series.index[-1], series.iloc[-1], "o",
            color="#4682B4", markersize=2)     # latest value
    ax.set_ylim(low - 2, high + 2)           # shared scale
    ax.axis("off")

In [ ]:
# A table of states with a sparkline per row, all on one vertical scale
order = oxxo_state.sort_values("per10k", ascending=False)
low, high = retail.values.min(), retail.values.max()
fig = plt.figure(figsize=(WIDTH, 3.2))
for col, rows in enumerate([order.iloc[:16], order.iloc[16:]]):
    x0 = 0.02 + 0.5 * col
    for x, text, ha in [(x0, "State", "left"), (x0 + 0.23, "OXXO per 10k", "right"),
                        (x0 + 0.255, "Real retail, 2018–25", "left"), (x0 + 0.48, "Latest", "right")]:
        fig.text(x, 0.955, text, fontsize=5.5, color="#555555", ha=ha)
    for i, (_, s) in enumerate(rows.iterrows()):
        top, series = 0.92 - 0.056 * i, retail[s.NOM_ENT]
        fig.text(x0, top - 0.02, s["name"], fontsize=6, va="center")
        bar = fig.add_axes([x0 + 0.15, top - 0.04, 0.05, 0.04])            # density as a tiny bar
        bar.barh(0, s.per10k, color=STEEL, height=0.55); bar.set_xlim(0, 4.3); bar.axis("off")
        fig.text(x0 + 0.23, top - 0.02, f"{s.per10k:.1f}", fontsize=5.5, va="center", ha="right")
        spark = fig.add_axes([x0 + 0.255, top - 0.045, 0.17, 0.05])        # the sparkline
        spark.plot(series.index, series.values, color="#9A9A9A", lw=0.7)
        spark.plot(series.index[-1], series.iloc[-1], "o", color=STEEL, ms=2)
        spark.plot(series.idxmin(), series.min(), "o", color="#555555", ms=1.5)
        spark.set_ylim(low - 2, high + 2); spark.axis("off")
        fig.text(x0 + 0.48, top - 0.02, f"{series.iloc[-1]:.0f}", fontsize=5.5, va="center", ha="right", color=STEEL)
save(fig, "fig12_sparklines")

In [ ]:
# Year-on-year change in each state's annual average,
# for the years before the 2018 base
annual = emec.groupby(["ENTIDAD", "ANIO"])["M000W"].mean()
change = 100 * annual.groupby(level="ENTIDAD").pct_change()
years = change.index.get_level_values("ANIO")
early = change[(years >= 2009) & (years <= 2018)]
print(early[early > 25].round(0).astype(int).to_string())

In [ ]:
from matplotlib.patches import Rectangle

# Each state as one tile, at a (column, row) near its place on
# the map; NL is Nuevo Leon, MEX the Estado de Mexico
TILES = {"BC": (0, 0), "SON": (1, 0), "CHIH": (2, 0),
         "COAH": (3, 0), "NL": (4, 0), "BCS": (0, 1),
         "SIN": (1, 1), "DGO": (2, 1), "ZAC": (3, 1),
         "SLP": (4, 1), "TAM": (5, 1), "NAY": (1, 2),
         "AGS": (2, 2), "GTO": (3, 2), "QRO": (4, 2),
         "HGO": (5, 2), "VER": (6, 2), "YUC": (9, 2),
         "JAL": (1, 3), "MICH": (2, 3), "MEX": (3, 3),
         "CDMX": (4, 3), "TLAX": (5, 3), "PUE": (6, 3),
         "TAB": (7, 3), "CAMP": (8, 3), "QROO": (9, 3),
         "COL": (1, 4), "GRO": (3, 4), "MOR": (4, 4),
         "OAX": (6, 4), "CHIS": (7, 4)}

def draw_tile(ax, code, rate, series, shares):
    col, row = TILES[code]
    x, y, size = col, -row, 0.93
    fill = mpl.colormaps["cividis"]((rate - 0.5) / 3.7)
    ax.add_patch(Rectangle((x, y), size, size, color=fill))
    ax.text(x + 0.05, y + size - 0.05, code, va="top")
    t = np.linspace(x + 0.08, x + size - 0.08, len(series))
    height = (series - low) / (high - low)   # 0 to 1
    ax.plot(t, y + 0.24 + 0.38 * height)
    left = x                             # chain shares strip
    for chain, color in CHAIN.items():
        width = shares[chain] / 100 * size
        ax.add_patch(Rectangle((left, y), width, 0.15,
                               color=color))
        left += width

In [ ]:
# Each state as one equal square, placed roughly where it sits on the map
TILES = {"BC": (0, 0), "SON": (1, 0), "CHIH": (2, 0), "COAH": (3, 0), "NL": (4, 0),
         "BCS": (0, 1), "SIN": (1, 1), "DGO": (2, 1), "ZAC": (3, 1), "SLP": (4, 1), "TAM": (5, 1),
         "NAY": (1, 2), "AGS": (2, 2), "GTO": (3, 2), "QRO": (4, 2), "HGO": (5, 2), "VER": (6, 2), "YUC": (9, 2),
         "JAL": (1, 3), "MICH": (2, 3), "MEX": (3, 3), "CDMX": (4, 3), "TLAX": (5, 3), "PUE": (6, 3), "TAB": (7, 3),
         "CAMP": (8, 3), "QROO": (9, 3), "COL": (1, 4), "GRO": (3, 4), "MOR": (4, 4), "OAX": (6, 4), "CHIS": (7, 4)}
ABBREVIATION = dict(zip([f"{i:02d}" for i in range(1, 33)],
    ["AGS", "BC", "BCS", "CAMP", "COAH", "COL", "CHIS", "CHIH", "CDMX", "DGO", "GTO", "GRO", "HGO", "JAL", "MEX", "MICH",
     "MOR", "NAY", "NL", "OAX", "PUE", "QRO", "QROO", "SLP", "SIN", "SON", "TAB", "TAM", "TLAX", "VER", "YUC", "ZAC"]))
norm, cmap, SIZE = mpl.colors.Normalize(0.5, 4.2), mpl.colormaps["cividis"], 0.93
low, high = retail.values.min(), retail.values.max()

fig = plt.figure(figsize=(WIDTH, 3.15)); ax = fig.add_axes([0, 0, 1, 1])
for _, s in oxxo_state.iterrows():
    col, row = TILES[ABBREVIATION[s.cve_ent]]; x, y = col, -row
    fill = cmap(norm(s.per10k))
    ink = "black" if 0.299 * fill[0] + 0.587 * fill[1] + 0.114 * fill[2] > 0.5 else "white"   # readable text
    ax.add_patch(Rectangle((x, y), SIZE, SIZE, fc=fill, ec="none"))
    ax.text(x + 0.05, y + SIZE - 0.05, ABBREVIATION[s.cve_ent], fontsize=6.5, fontweight="bold", color=ink, va="top")
    ax.text(x + SIZE - 0.05, y + SIZE - 0.05, f"{s.per10k:.1f}", fontsize=6.5, color=ink, va="top", ha="right")
    series = retail[s.NOM_ENT]                                            # sparkline, same scale in every tile
    ax.plot(np.linspace(x + 0.08, x + SIZE - 0.08, len(series)), y + 0.24 + (series.values - low) / (high - low) * 0.38,
            color=ink, lw=0.8)
    left = x                                                               # chain composition strip
    for chain in CHAIN_ORDER:
        width = shares_by_code.loc[s.cve_ent, chain] / 100 * SIZE
        ax.add_patch(Rectangle((left, y), width, 0.15, fc=CHAIN[chain], ec="none")); left += width
ax.set(xlim=(-0.05, 10), ylim=(-4.05, 0.97)); ax.set_aspect("equal"); ax.axis("off")
cax = fig.add_axes([0.63, 0.85, 0.34, 0.035])
bar = fig.colorbar(mpl.cm.ScalarMappable(norm, cmap), cax=cax, orientation="horizontal")
bar.set_label("OXXO stores per 10,000 residents", fontsize=7); bar.outline.set_linewidth(0.3)
fig.text(0.63, 0.70, "Line: real retail revenue, 12-month average,\nDec 2018–Mar 2025, same scale in every tile",
         fontsize=6.5, color="#444444", va="top")
fig.legend([Rectangle((0, 0), 1, 1, color=CHAIN[c]) for c in CHAIN_ORDER], CHAIN_ORDER, loc="upper left",
           bbox_to_anchor=(0.795, 0.215), ncol=2, frameon=False, fontsize=6.5, handlelength=0.9, handleheight=0.9,
           columnspacing=0.6, handletextpad=0.35, labelspacing=0.3, title="Strip: chain shares", title_fontsize=6.5,
           alignment="left", borderaxespad=0)
save(fig, "fig12_tilemap")